In [2]:
import json
import os
from datasets import load_dataset
from pydantic import BaseModel

import vertexai
from vertexai.generative_models import GenerativeModel
import google.colab.auth # Import auth module

# --- Configuration Options ---
use_gemini_api_key = False # Set to True to use Gemini API Key, False to use Vertex AI

# 1. Initialize API client using Vertex AI
# Replace 'YOUR_PROJECT_ID' and 'YOUR_REGION' with your actual Google Cloud Project ID and desired region
project_id = '' # e.g., 'my-gcp-project-12345'
region = '' # e.g., 'us-central1'

# Or Use a Gemini API key
# IMPORTANT: Make sure to replace 'YOUR_GEMINI_API_KEY' with your actual key
# You can get an API key from Google AI Studio: https://makersuite.google.com/app/apikey
gemini_api_key = 'YOUR_GEMINI_API_KEY'


if use_gemini_api_key:
    import google.generativeai as genai
    genai.configure(api_key=gemini_api_key)
    client = genai.GenerativeModel('gemini-pro') # Using 'gemini-pro' for non-Vertex AI access
    print("Using Gemini API key for client initialization.")
else:
    # Authenticate user to Google Cloud
    google.colab.auth.authenticate_user()
    vertexai.init(project=project_id, location=region)
    client = GenerativeModel('gemini-2.5-flash') # Use the GenerativeModel directly from Vertex AI
    print("Using Vertex AI for client initialization.")

# Define the expected JSON schema for strict parsing
class GoTranslation(BaseModel):
    go_prompt: str
    go_solution: str
    go_test_harness: str

# 2. Load the dataset
# Note: Replace 'TIGER-Lab/AceCode-87K' with the exact split/config you need
print("Loading dataset...")
dataset = load_dataset("TIGER-Lab/AceCode-87K", split="train")

def translate_to_go(example):
    prompt_template = f"""
    You are an expert polyglot programmer. Translate this Python problem to Go.

    PROMPT: {example['question']}
    REFERENCE SOLUTION: {example.get('solution', 'N/A')}
    TEST CASES: {example.get('test_cases', 'N/A')}

    Return a JSON with keys: go_prompt, go_solution, go_test_harness.
    The go_test_harness MUST be a runnable 'package main' with a 'func main()' that panics on failure. Do not include the solution function inside the test harness.
    """

    try:
        response = client.generate_content(
            prompt_template,
            generation_config={
                'response_mime_type': 'application/json',
            },
        )

        # Parse the JSON response and validate with Pydantic
        raw_result = json.loads(response.text)
        result = GoTranslation.parse_obj(raw_result)

        return {
            "input": result.go_prompt,
            "output": "placeholder", # As requested
            "extra_env_info": {"test_code": result.go_test_harness},
            "task_name": "go_verify_task", # As requested
            "go_solution": result.go_solution, # Keep go_solution for potential future use
            "translation_status": "success"
        }
    except Exception as e:
        print(f"Error translating row: {e}")
        return {
            "input": "", "output": "placeholder", "extra_env_info": {"test_code": ""}, "task_name": "go_verify_task",
            "go_solution": "", "translation_status": "failed"
        }

# 3. Apply the translation to a small subset first to test
print("Translating first 5 rows...")
subset = dataset.select(range(20))
translated_dataset = subset.map(translate_to_go)

# 4. Save your new Go dataset
translated_dataset.to_json("AceCode-87K-Go-Subset.jsonl")
print("Saved to AceCode-87K-Go-Subset.jsonl")

/usr/local/lib/python3.12/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Using Vertex AI for client initialization.
Loading dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00011.parquet:   0%|          | 0.00/92.2M [00:00<?, ?B/s]

data/train-00001-of-00011.parquet:   0%|          | 0.00/91.4M [00:00<?, ?B/s]

...

Translating first 5 rows...


Map:   0%|          | 0/20 [00:00<?, ? examples/s]

/tmp/ipykernel_411/1983556152.py:69: PydanticDeprecatedSince20: The `parse_obj` method is deprecated; use `model_validate` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  result = GoTranslation.parse_obj(raw_result)


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved to AceCode-87K-Go-Subset.jsonl


In [3]:
import json

file_path = 'AceCode-87K-Go-Subset.jsonl'
num_lines_to_display = 20

print(f"Displaying first {num_lines_to_display} lines from '{file_path}':")
try:
    with open(file_path, 'r') as f:
        for i, line in enumerate(f):
            if i >= num_lines_to_display:
                break
            print(line.strip())
except FileNotFoundError:
    print(f"Error: File '{file_path}' not found.")
except Exception as e:
    print(f"An error occurred: {e}")

Displaying first 20 lines from 'AceCode-87K-Go-Subset.jsonl':
{"id":"oss_0","source":"oss","question":"Given a list of strings, implement a function `find_palindromes` that returns a new list containing only the strings that are palindromes. A palindrome is defined as a word that reads the same forward and backward, ignoring case and spaces. For example, if the input list is `['radar', 'hello', 'level', 'world', 'Anna']`, the function should return `['radar', 'level', 'Anna']`. Your function should handle empty strings and variations in case appropriately.","test_cases":["assert find_palindromes(['radar', 'hello', 'level', 'world', 'Anna']) == ['radar', 'level', 'Anna']","assert find_palindromes(['racecar', 'civic', 'deified', 'hello', 'world']) == ['racecar', 'civic', 'deified']","assert find_palindromes(['noon', 'test', 'rotor', 'python']) == ['noon', 'rotor']","assert find_palindromes(['']) == ['']","assert find_palindromes(['Able was I ere I saw Elba', 'Hello']) == ['Able was I ere

In [4]:
import json

input_file = 'AceCode-87K-Go-Subset.jsonl'
output_file = 'AceCode-87K-Go-Cleaned.jsonl'

print(f"Processing '{input_file}' to create '{output_file}'...")

try:
    with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
        for line in infile:
            # Parse the original JSON object
            original_data = json.loads(line)

            # Extract only the desired fields
            cleaned_data = {
                "input": original_data.get("input", ""),
                "output": original_data.get("output", "placeholder"),
                "extra_env_info": original_data.get("extra_env_info", {}),
                "task_name": original_data.get("task_name", "go_verify_task")
            }

            # Write the cleaned JSON object to the new file
            outfile.write(json.dumps(cleaned_data) + '\n')
    print(f"Successfully created '{output_file}' with filtered data.")
except FileNotFoundError:
    print(f"Error: Input file '{input_file}' not found.")
except Exception as e:
    print(f"An error occurred during processing: {e}")

Processing 'AceCode-87K-Go-Subset.jsonl' to create 'AceCode-87K-Go-Cleaned.jsonl'...
Successfully created 'AceCode-87K-Go-Cleaned.jsonl' with filtered data.


After running the above cell, you can inspect the newly created `AceCode-87K-Go-Cleaned.jsonl` file. Here's how to display its first few lines to verify the format:

In [5]:
import json

file_path = 'AceCode-87K-Go-Cleaned.jsonl'
num_lines_to_display = 25

print(f"Displaying first {num_lines_to_display} lines from '{file_path}':")
try:
    with open(file_path, 'r') as f:
        for i, line in enumerate(f):
            if i >= num_lines_to_display:
                break
            print(line.strip())
except FileNotFoundError:
    print(f"Error: File '{file_path}' not found.")
except Exception as e:
    print(f"An error occurred: {e}")

Displaying first 25 lines from 'AceCode-87K-Go-Cleaned.jsonl':
{"input": "Given a slice of strings, implement a function `findPalindromes` that returns a new slice containing only the strings that are palindromes. A palindrome is defined as a word that reads the same forward and backward, ignoring case, spaces, and punctuation. For example, if the input slice is `['radar', 'hello', 'level', 'world', 'Anna']`, the function should return `['radar', 'level', 'Anna']`. Your function should handle empty strings and variations in case appropriately.", "output": "placeholder", "extra_env_info": {"test_code": "package main\n\nimport (\n\t\"fmt\"\n\t\"reflect\"\n)\n\n// The functions `isPalindrome` and `findPalindromes` are assumed\n// to be defined externally in the same package and are available for testing.\n\n// assertEqual is a helper function to compare two string slices.\n// It panics if the slices are not deeply equal, indicating a test failure.\nfunc assertEqual(name string, actual, ex